## Setup

In [60]:
#%pip install --upgrade symbolica
#%pip install --upgrade ipywidgets
from symbolica import *
from IPython.display import Markdown as md, display as dp
from ipywidgets import interact, interactive, fixed, interact_manual
import ipywidgets as widgets

def display(self) -> None:
    dp(md(self.to_latex().replace('gamma', '\\gamma').replace('mu', '\\mu')))

def display_graph(self: Graph) -> None:
    dp(md('```mermaid\n' + self.to_mermaid() + '\n```'))


set_license_key('gcolab-demo-key-do-not-use-elsewhere')

In [61]:
%%html
<style>
.cell-output-ipywidget-background {
    background-color: transparent !important;
}
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
</style>

## Expressions

Parse expressions with `E`:

In [67]:
e = E('(x+1)(y+1)+f(x,y)^2')
print(e)

f(x,y)^2+(x+1)*(y+1)


Build expressions by defining symbols with `S` and combining them:

In [71]:
x, f, y = S('x', 'f', 'y')
e = ((1+x)**2 + (1+y)**4).expand(x)
display(e)

$$2 x+x^{2}+\left(y+1\right)^{4}+1$$

Print modes:

In [5]:
print(e.format())

x+f(x,1)²


Parse numbers with `N`:

In [74]:
e = N(0.1, 0.01) * x * f(x,1)
print(e)

1/10*x*f(x,1)


Loosely precision-tracking floats:

In [7]:
print(N('1.234563`6'))
print(N('1.234`6'))
print(N('1.23456`6') - N('1.234`6'))

1.23456
1.23400
5.61e-4


Symbols have a namespace:

In [75]:
e = E('trace::x') + x
print(e)
print(e.format(show_namespaces=True))

x+x
x+trace::x


Assign attributes to symbols:

In [76]:
x, y = S('x', 'y')
dot = S('dot',is_symmetric=True, is_linear=True)
e = dot(x+y,3*x)
display(e)

$$3 dot\!\left(x,x\right)+3 dot\!\left(x,y\right)$$

### Operations on expressions

In [77]:
y=S('y')
e = 1+(1+x)/((1+x)**2).expand()
print(e)
display(e.cancel())
display(e.factor())

(x+1)*(2*x+x^2+1)^-1+1


$$\frac{1}{x+1}+1$$

$$\frac{1}{x+1} \left(x+2\right)$$

In [11]:
c = S('c')
x_r, y_r = Expression.solve_linear_system([f(c)*x + y + c, y + c**2], [x, y])
print('x =', x_r, ', y =', y_r)

x = (-c+c^2)*f(c)^-1 , y = -c^2


In [12]:
e = E('x+1')
e.save('test') # save in compressed binary format
e2 = Expression.load('test')
print(e2)

x+1


Canonize tensors:

In [13]:
g = S('g', is_symmetric=True)
fc = S('fc', is_cyclesymmetric=True)
mu, k = S('mu', 'k')

e = g(mu(2), mu(3))*fc(mu(4), mu(2), k, mu(4), k, mu(3))-g(mu(4),mu(3))*fc(k,mu(2),k,mu(3),mu(2),mu(4))
print(e)
print(e.canonize_tensors([(mu(1), 0), (mu(2), 0), (mu(3), 0), (mu(4), 0)]))

g(mu(2),mu(3))*fc(k,mu(3),mu(4),mu(2),k,mu(4))-g(mu(3),mu(4))*fc(k,mu(2),k,mu(3),mu(2),mu(4))
0


## Series expansion

In [79]:
s = E('sin(x)/(1-x)').series(x, 0, 5).exp()
print(s)

1+x+3/2*x^2+2*x^3+65/24*x^4+𝒪(x^5)


In [15]:
@interact(depth=(1,20))
def update_series(depth=4):
    e = E('sin(x)/(1-x)').series(S('x'), 0, depth).exp()
    return md(e.to_latex())

interactive(children=(IntSlider(value=4, description='depth', max=20, min=1), Output()), _dom_classes=('widget…

### Matrices

In [80]:
t = S('t')

a = Matrix.from_nested([[t, 2], [3, 4]])
display(a.inv() + a)

$$\begin{pmatrix}\frac{2-3t+2t^{2}}{-3+2t} & \frac{-7+4t}{-3+2t} \\ \frac{-21+12t}{-6+4t} & \frac{-24+17t}{-6+4t}\end{pmatrix}$$

In [17]:
b = Matrix.vec([1,3])
display(a*b)

$$\begin{pmatrix}6+t \\ 15\end{pmatrix}$$

In [18]:
display(a.solve(b))

$$\begin{pmatrix}\frac{-1}{-3+2t} \\ \frac{-3+3t}{-6+4t}\end{pmatrix}$$

### Polynomials

Any expression can be turned into a (rational) polynomial:

In [ ]:
a = E('x+ x* y + f(w)*x /z + 5').to_polynomial()

for v in a.get_var_list():
    print(v)


x
y
z^-1
f(w)


## Pattern matching

In [84]:
e = f(x**2, x, f(x))+x+2
display(e)
display(e.replace(x, y**2, level_range=(0,1)))

$$x+f\!\left(x^{2},x,f\!\left(x\right)\right)+2$$

$$y^{2}+f\!\left(y^{4},y^{2},f\!\left(x\right)\right)+2$$

In [21]:
y = S('y')
e = x*f(x, y, f(x, y))
e = e.replace(x, 5, level_range=(1,2))
display(e)

$$x f\!\left(5,y,f\!\left(5,y\right)\right)$$

Wildcards can match a single atom in the subexpression:

In [97]:
x_ = S('x_')
e = x*f(x, y, f(x, 2, y))
e = e.replace(x_, x_**2, x_.req(lambda x: x == y))
display(e)

$$x f\!\left(x,y^{2},f\!\left(x,2,y^{2}\right)\right)$$

A wildcard can be restricted:

In [90]:
x_, fact = S('x_', 'fact')

e = fact(10).replace(fact(x_), x_*fact(x_ - 1), x_ > 1, repeat=True).replace(fact(1), 1)
display(e)

$$3628800$$

No match here:

In [91]:
y_ , z= S('y_', 'z')
e = f(x*y*z)
e = e.replace(f(x_*x), 1)
display(e)

$$f\!\left(x y z\right)$$

A single `_` means it matches a single atom completely. `__` means it can match a non-empty subset and `___` means match empty subsets as well.

In [93]:
x__, x___, y_, y__, y___ = S(
    'x__', 'x___', 'y_', 'y__', 'y___')
e = f(*[i for i in range(6)])

for r in e.replace_iter(f(x__, x_, y___), f(x__, 100, y___)):
    print(r)

f(0,1,2,3,4,100)
f(0,1,2,3,100,5)
f(0,1,2,100,4,5)
f(0,1,100,3,4,5)
f(0,100,2,3,4,5)


Match without replacement:

In [94]:
for r in e.match(f(x___, x_, y___)):
    print(r[x_])

5
4
3
2
1
0


Lists are wrapped in the `arg` function:

In [96]:
e = f(1,2,3,4)
a = e.replace(f(x__), x__)
print(a)

f(a)

arg(1,2,3,4)


f(1,2,3,4)

The right-hand side can also be a function that takes the match dictionary as arguments:

In [28]:
k, x_ = S('k', 'x_')

r = (k(1)).replace(k(x_), lambda m: E("k(mu{})".format(m[x_])))
display(r)

$$k\!\left(\mu1\right)$$

### Examples
- Check if a graph is connected by shrinking edges

In [98]:
vs = S('vs', is_symmetric=True)
x_, l1___, l2___, r1___, r2___ = S('x_', 'l1___', 'l2___', 'r1___', 'r2___')
g = vs(0, 1, 4)*vs(1, 2, 5)*vs(0, 2, 3)*vs(3, 4, 5) # Mercedes topology

g = g.replace(vs(l1___,x_,r1___)*vs(l2___,x_,r2___),
        vs(l1___,r1___,l2___,r2___), repeat=True)
print(g)

vs(2,2,4,4,5,5)


- Write code that applies the gamma chain relation:
$$
\gamma(\mu_1,\ldots,\nu,\nu,\ldots\mu_n) = D \gamma(\mu_1,\ldots,\mu_n)\\
\gamma(\mu_1,\ldots,\nu_2,\nu_1,\ldots\mu_n) = 2g(\nu_1,\nu_2) \gamma(\mu_1,\ldots,\mu_n) - \gamma(\mu_1,\ldots,\nu_1,\nu_2,\ldots\mu_n) ; \nu_2 > \nu_1
$$

In [30]:
mu = S(*['mu_' + str(i) for i in range(10)])
gamma, D = S('gamma', 'D')
g = S('g', is_symmetric=True)

e = gamma(mu[0],mu[1],mu[0], mu[2], mu[4], mu[4], mu[3])

e = e.replace(gamma(x___, x_, x_, y___), D*gamma(x___,y___), repeat=True)
e = e.replace(gamma(x___, x_, y_, y___), 2*g(x_,y_)*gamma(x___, y___) - gamma(x___, y_, x_, y___), x_.req_cmp_gt(y_, True), repeat=True).expand()

display(e)

$$-D \gamma\!\left(\mu_0,\mu_0,\mu_1,\mu_2,\mu_3\right)+2 D g\!\left(\mu_0,\mu_1\right) \gamma\!\left(\mu_0,\mu_2,\mu_3\right)$$

## Graph generation

In [31]:
g, q = S('gl', 'q')

external= [(1, (None, g)), (2, (None, g))]
vertices =  [((None, g), (None, g), (None, g), (None, g)),
                             ((None, g), (None, g), (None, g)),
                            ((None, g), (True, q), (False, q))]

graphs = graphs = Graph.generate(external, vertices, max_loops=1, max_bridges=0)

for graph, sym in graphs.items():
    print(f"Symmetry factor: 1/{sym}")
    display_graph(graph)

Symmetry factor: 1/1


```mermaid
graph TD;
  0["0"];
  1["0"];
  2["1"];
  3["2"];
  0 -->|"q"| 1;
  0 ---|"gl"| 3;
  1 -->|"q"| 0;
  1 ---|"gl"| 2;

```

Symmetry factor: 1/2


```mermaid
graph TD;
  0["0"];
  1["0"];
  2["1"];
  3["2"];
  0 ---|"gl"| 1;
  0 ---|"gl"| 1;
  0 ---|"gl"| 3;
  1 ---|"gl"| 2;

```

## Expression evaluation

One-time evaluation

In [32]:
print((x+5).evaluate({x: 5.}, {}))

10.0


Evaluate nested expressions

\begin{align*}
e &= x + \pi + \cos(x) + f(g(x+1),2x)\\
f(y,z) &=y^2 + z^2 y^2\\
g(y) &=x+y+5
\end{align*}

In [33]:
x, y, z, pi, f, g = S(
    'x', 'y', 'z', 'pi', 'f', 'g')

e1 = E("x + pi + cos(x) + f(g(x+1),x*2)")
fd = E("y^2 + z^2*y^2")
gd = E("x + y + 5")

ev = e1.evaluator({pi: N(22)/7},
             {(f, "f", (y, z)): fd,
              (g, "g", (y, )): gd},
            [x])
res = ev.evaluate([[5.], [6.]])
print(res)

[[25864.42651932832], [46990.103027429504]]


### Evaluating a large expression

In [99]:
f13 = E(open('f13.txt').read())

One-time evaluation:

In [100]:
vars = ["alpha", "amuq", "ammu", "xcp1", "e1245", "xcp4", "e3e2", "e1234", "e2345", "e1235",
        "e1345", "amel2", "e2e1", "e5e2", "e4e2", "e3e1", "e4e1", "e5e1", "ammu2", "amuq2", "e5e3",
        "e4e3", "x5", "x6", "x1", "x3", "x4", "xcp3", "xcp2"]

%timeit r = f13.evaluate({S(x): float(i) for i, x in enumerate(vars)}, {})

16.4 ms ± 130 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [101]:
evaluator = f13.evaluator({}, {}, [S(x) for x in vars], iterations=1)

In [102]:
%timeit evaluator.evaluate([[float(i) for i in range(len(vars))]])

89.9 μs ± 1.14 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


Compile instantly with custom inline assembly

In [103]:
compiled_evaluator = evaluator.compile('f13', 'f13.cpp', 'f13.so', )

In [104]:
%timeit compiled_evaluator.evaluate([[float(i) for i in range(len(vars))]])

19.4 μs ± 36.6 ns per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [105]:
evaluator.get_instructions()

([('mul', ('out', 0), [('param', 0), ('param', 14)]),
  ('mul', ('temp', 1), [('param', 9), ('param', 14)]),
  ('mul', ('temp', 2), [('param', 9), ('const', 16)]),
  ('mul', ('temp', 3), [('param', 14), ('param', 14)]),
  ('mul', ('temp', 4), [('const', 43), ('temp', 3)]),
  ('mul', ('temp', 5), [('const', 15), ('temp', 3)]),
  ('mul', ('temp', 6), [('const', 18), ('temp', 3)]),
  ('mul', ('temp', 7), [('const', 37), ('temp', 3)]),
  ('mul', ('temp', 8), [('const', 44), ('temp', 3)]),
  ('mul', ('temp', 9), [('const', 24), ('temp', 3)]),
  ('mul', ('temp', 10), [('const', 40), ('temp', 3)]),
  ('mul', ('temp', 11), [('param', 0), ('param', 28)]),
  ('mul', ('temp', 12), [('param', 0), ('param', 17)]),
  ('mul', ('temp', 13), [('param', 0), ('param', 27)]),
  ('mul', ('temp', 14), [('param', 0), ('param', 13)]),
  ('mul', ('temp', 15), [('param', 9), ('param', 28)]),
  ('mul', ('temp', 16), [('param', 9), ('param', 18)]),
  ('mul', ('temp', 17), [('param', 9), ('param', 27)]),
  ('mul',

## Transformers

Transformers are like functions that execute internally and take an expression and yield another expression.

In [109]:
t = Transformer().expand().collect(x)
e = E('(1+x)^2')
r = t(e)
print(r)

2*x+x^2+1


Transformers can be bound to an expression:

In [110]:
t = ((x+1)**2).transform().expand()
t.execute()

2*x+x^2+1

The following code tries to expand `x_`, but it does not work:

In [112]:
x, f, x_ = S('x', 'f', 'x_')
e = f((x+1)**2)
e = e.replace(f(x_), f(x_.transform().expand()))
display(e)

$$f\!\left(2 x+x^{2}+1\right)$$

Expansion is executed after substituting the matched value of `x_`, since `replace` calls the transformer when it matched

In [43]:
def some_function(x):
    print('Hello from Python!')
    return x**2


e = f((x+1)**2)
e = e.replace(f(x_), x_.transform().map(some_function))
display(e)

Hello from Python!


$$\left(x+1\right)^{4}$$

Transformers can be used to create a computational graph: a series of instructions to execute

In [113]:
T = Transformer()  # short-hand

e = f(10)
tr = T.repeat(
    T.expand(),
    T.replace(f(x_), f(x_ - 1) + f(x_ - 2), x_ > 1),
)

display(tr(e))

$$34 f\!\left(0\right)+55 f\!\left(1\right)$$

Transformers support branching:

In [45]:
t = T.map_terms(T.if_changed(T.replace(x, y), T.print()))
print(t(x + y + 4))

y
2*y+4


In [46]:
t = T.map_terms(T.repeat(
    T.replace(y, 4),
    T.if_changed(T.replace(x, y),
                 T.break_chain()),
    T.print()  # print of y is never reached
))
t(x)

4
4


4

## Custom normalization

Create a function that is linear in the variable $n$

In [47]:
n, x_, y, y_ = S('n', 'x_', 'y', 'y_')
dot2_ = S('dotp_', is_symmetric=True, is_linear=True)
dot2 = S('dot2', is_symmetric=True, is_linear=True, custom_normalization=Transformer().replace(dot2_(n*x_,y_), n*dot2_(x_, y_)))

In [48]:
print(E('dot2(n*x,x + n*y)'))

n*dot2(x,x)+n^2*dot2(x,y)


## Streams

Terms can be streamed to and from disk automatically:

In [ ]:
x, y = S('x', 'y')
#e = ((x+1)**20).expand()
stream = TermStreamer(n_cores=4)
stream.push(x)
stream.push(y)
stream = stream.map(Transformer().replace(x, y))

stream.normalize()
stream.map(Transformer().print())

e = stream.to_expression()
print('normalized', e)

2*y
normalized 2*y


-----

## Exercises

$$
I(n) = I(n-1) \frac{2 + (4-2 \epsilon) - 2  n}{2(n - 1) m^2}\,; n \geq 1
$$


In [50]:
ep, m, n_ = S('ep', 'm', 'n_')
topo = S('topo')
T = Transformer()

e = E("topo(10)")
e = e.replace(topo(n_),
                        topo(n_ - 1) * (2 + (4-2*ep) - 2 * n_) /
                        (2*(n_ - 1)) / m**2,
                        n_.req_gt(1), repeat=True).expand()
display(e)

$$\frac{1}{72} ep m^{-18} topo\!\left(1\right)+\frac{223}{10080} ep^{2} m^{-18} topo\!\left(1\right)+\frac{1}{5670} ep^{3} m^{-18} topo\!\left(1\right)-\frac{101}{5760} ep^{4} m^{-18} topo\!\left(1\right)-\frac{229}{17280} ep^{5} m^{-18} topo\!\left(1\right)-\frac{13}{2880} ep^{6} m^{-18} topo\!\left(1\right)-\frac{7}{8640} ep^{7} m^{-18} topo\!\left(1\right)-\frac{1}{13440} ep^{8} m^{-18} topo\!\left(1\right)-\frac{1}{362880} ep^{9} m^{-18} topo\!\left(1\right)$$

In [51]:
e = E("topo(10)")
e = e.replace(topo(n_),
                        topo(n_ - 1) * Expression.COEFF(2 + (4-2*ep) - 2 * n_) /
                        (2*(n_ - 1)) / m**2,
                        n_.req_gt(1), repeat=True)
display(e)

$$[\frac{5040ep+8028ep^{2}+64ep^{3}-6363ep^{4}-4809ep^{5}-1638ep^{6}-294ep^{7}-27ep^{8}-ep^{9}}{362880}] m^{-18} topo\!\left(1\right)$$

1. Write code that applies the trace rule
$$Tr(\mu_1,\ldots,\mu_n) = \sum_{k=2}^n (-1)^k g(\mu_1,\mu_k) Tr(\mu_2,\ldots,\mu_{k-1},\mu_{k+1},\ldots \mu_n)$$

In [52]:
from symbolica import Expression

p = S(*['p_' + str(i + 1) for i in range(10)])
p_ = S(*['p_' + str(i + 1) + '_' for i in range(10)])
g = S('g', is_symmetric=True)

e = f(*p[:6])
display(e)

$$f\!\left(p_1,p_2,p_3,p_4,p_5,p_6\right)$$

Here is the FORM code that does the trace:

```
#-
off statistics;
V p,p0,p1,...,p20;
CF f;
CF d(s);

L F = f(p1,...,p10);

#do i = 10,1,-2
    id f(p1?,...,p`i'?) = 
        #if `i' > 2
            + d(p1, p2) * f(p3,...,p`i')
        #else
            + d(p1,p2)
        #endif
        
        #do k = 3,{`i'-1},1
            + (-1)^(`k'+1) * d(p1, p`k') * f(p2,...,p{`k'-1},p{`k'+ 1},...,p`i')
        #enddo

        #if `i' > 2
        + (-1)^(`i') * d(p1, p`i') * f(p2,...,p{`i'-1})
        #endif
    ;
    .sort:trace-`i';
#enddo

id f = 4;
id f(?a) = 0;

Print +s;
.end
```

2. Apply the gluon Feynman rule

In [53]:
k = S('mom', is_symmetric=True)
vx, coll = S('vx', 'coll')
nu = S(*['nu%d' % i for i in range(3)])
mu = S(*['mu%d' % i for i in range(12)])
d = S('d', is_symmetric=True, is_linear=True)
v = S('v', is_linear=True)
k_ = S(*['k%d_' % i for i in range(5)])
mu_ = S(*['mu%d_' % i for i in range(5)])

x_, y_, x__, x___, y___ = S('x_', 'y_', 'x__', 'x___', 'y___')

# k(0) is external momentum
r = vx(-k(0), k(0)-k(1), k(1), nu[1], mu[1], mu[8]) *\
    vx(-k(1), k(2), k(1)-k(2), mu[1], mu[2], mu[9]) *\
    vx(-k(2), k(3), k(2)-k(3), mu[2], mu[3], mu[10]) *\
    vx(-k(3), k(4), k(3)-k(4), mu[3], mu[4], mu[11]) *\
    vx(-k(4), k(0), k(4)-k(0), mu[4], nu[2], mu[5]) *\
    vx(-k(4)+k(0), -k(3)+k(4), k(3)-k(0), mu[5], mu[11], mu[6]) *\
    vx(-k(3)+k(0), -k(2)+k(3), k(2)-k(0), mu[6], mu[10], mu[7]) *\
    vx(-k(2)+k(0), -k(1)+k(2), k(1)-k(0), mu[7], mu[9], mu[8])

r2 = r.replace(vx(k_[1], k_[2], k_[3], mu_[1], mu_[2], mu_[3]),
                   (- d(mu_[1], mu_[3]) * d(k_[1], mu_[2])
                    + d(mu_[1], mu_[2]) * d(k_[1], mu_[3])
                    + d(mu_[2], mu_[3]) * d(k_[2], mu_[1])
                    - d(mu_[1], mu_[2]) * d(k_[2], mu_[3])
                    - d(mu_[2], mu_[3]) * d(k_[3], mu_[1])
                    + d(mu_[1], mu_[3]) * d(k_[3], mu_[2])
                    )
                   )

print(r2.format(terms_on_new_line=True))

((-d(nu2,mom(0))+d(nu2,mom(4)))*d(mu4,mu5)-(-d(mu4,mom(0))+d(mu4,mom(4)))*d(nu2,mu5)-d(nu2,mu4)*d(mu5,mom(0))-d(nu2,mu4)*d(mu5,mom(4))+d(nu2,mu5)*d(mu4,mom(0))+d(mu4,mu5)*d(nu2,mom(4)))*((d(nu1,mom(0))-d(nu1,mom(1)))*d(mu1,mu8)-(d(mu8,mom(0))-d(mu8,mom(1)))*d(mu1,nu1)-d(mu1,nu1)*d(mu8,mom(0))-d(mu1,mu8)*d(nu1,mom(1))+d(nu1,mu8)*d(mu1,mom(0))+d(nu1,mu8)*d(mu1,mom(1)))*(-(-d(mu5,mom(0))+d(mu5,mom(3)))*d(mu6,mu11)+(-d(mu5,mom(3))+d(mu5,mom(4)))*d(mu6,mu11)-(-d(mu6,mom(3))+d(mu6,mom(4)))*d(mu5,mu11)+(-d(mu11,mom(0))+d(mu11,mom(3)))*d(mu5,mu6)+(d(mu6,mom(0))-d(mu6,mom(4)))*d(mu5,mu11)-(d(mu11,mom(0))-d(mu11,mom(4)))*d(mu5,mu6))*(-(-d(mu6,mom(0))+d(mu6,mom(2)))*d(mu7,mu10)+(-d(mu6,mom(2))+d(mu6,mom(3)))*d(mu7,mu10)-(-d(mu7,mom(2))+d(mu7,mom(3)))*d(mu6,mu10)+(-d(mu10,mom(0))+d(mu10,mom(2)))*d(mu6,mu7)+(d(mu7,mom(0))-d(mu7,mom(3)))*d(mu6,mu10)-(d(mu10,mom(0))-d(mu10,mom(3)))*d(mu6,mu7))*(-(-d(mu7,mom(0))+d(mu7,mom(1)))*d(mu8,mu9)+(-d(mu7,mom(1))+d(mu7,mom(2)))*d(mu8,mu9)-(-d(mu8,mom(1))+d(mu8,

# Solutions

In [54]:

T = Transformer()
i1, i2 = S('i1', 'i2')
p = S(*['p' + str(i + 1) for i in range(10)])
p_ = S(*['p' + str(i + 1) + '_' for i in range(10)])
x_, y_, x___, y___, z___, i1_, i2_ = S(
    'x_', 'y_', 'x___', 'y___', 'z___', 'i1_', 'i2_')

gamma = S('γ')


# Canonicalize a gamma string
def gamma_canonicalize() -> Transformer:
    return T.repeat(
        T.expand(),
        T.replace(gamma(i1_, x___, x_, x_, y___, i2_),
                      d(x_, x_)*gamma(i1_, x___, y___, i2_)),
        T.replace(gamma(i1_, x___, x_, y_, z___, i2_), -gamma(i1_, x___, y_, x_, z___, i2_)+2*g(y_, x_)*gamma(i1_, x___, z___, i2_),
                      x_.req_cmp_gt(y_, True))
    )

In [55]:

# Perform the trace of a closed gamma string,
# working from the chain length `max_len` back to 0
def gamma_trace(max_len: int) -> Transformer:
    trace = T
    for l in range(max_len, 0, -1):
        if l % 2 == 1:
            trace = trace.replace(gamma(*p_[:l]), 0)  # odd trace is 0
            continue

        trace = trace.chain(
            T.replace(gamma(x___, x_, x_, y___),
                          d(x_, x_)*gamma(x___, y___)),
            T.replace(gamma(
                *p_[:l]), sum((-1)**(k+1) * d(p_[0], p_[k]) * gamma(*p_[1:k], *p_[k+1:l]) for k in range(1, l))).expand())
    return trace.replace(gamma(), 4)


e = gamma(i1, p[0], p[1], p[3], p[4], p[3], p[5], i2)
print('Canonicalization of', e)
e = e.transform().chain(gamma_canonicalize()).execute()
print('\t', e)


e = gamma(p[0], p[1], p[2], p[2], p[3], p[4], p[5], p[6])
print('Trace of', e)
e = e.transform().chain(gamma_trace(10)).execute()
print('\t', e.format(terms_on_new_line=True))

e = gamma(p[0], p[1], p[2], p[3]).transform().chain(gamma_trace(10)).execute()
print(e)

Canonicalization of γ(i1,p1,p2,p4,p5,p4,p6,i2)
	 2*g(p4,p5)*γ(i1,p1,p2,p4,p6,i2)-d(p4,p4)*γ(i1,p1,p2,p5,p6,i2)
Trace of γ(p1,p2,p3,p3,p4,p5,p6,p7)
	 4*d(p1,p2)*d(p3,p3)*d(p4,p5)*d(p6,p7)
-4*d(p1,p2)*d(p3,p3)*d(p4,p6)*d(p5,p7)
+4*d(p1,p2)*d(p3,p3)*d(p4,p7)*d(p5,p6)
-4*d(p1,p4)*d(p2,p5)*d(p3,p3)*d(p6,p7)
+4*d(p1,p4)*d(p2,p6)*d(p3,p3)*d(p5,p7)
-4*d(p1,p4)*d(p2,p7)*d(p3,p3)*d(p5,p6)
+4*d(p1,p5)*d(p2,p4)*d(p3,p3)*d(p6,p7)
-4*d(p1,p5)*d(p2,p6)*d(p3,p3)*d(p4,p7)
+4*d(p1,p5)*d(p2,p7)*d(p3,p3)*d(p4,p6)
-4*d(p1,p6)*d(p2,p4)*d(p3,p3)*d(p5,p7)
+4*d(p1,p6)*d(p2,p5)*d(p3,p3)*d(p4,p7)
-4*d(p1,p6)*d(p2,p7)*d(p3,p3)*d(p4,p5)
+4*d(p1,p7)*d(p2,p4)*d(p3,p3)*d(p5,p6)
-4*d(p1,p7)*d(p2,p5)*d(p3,p3)*d(p4,p6)
+4*d(p1,p7)*d(p2,p6)*d(p3,p3)*d(p4,p5)
4*d(p1,p2)*d(p3,p4)-4*d(p1,p3)*d(p2,p4)+4*d(p1,p4)*d(p2,p3)


### Gluon exercise

In [56]:
def gluon(r: Expression) -> Expression:
    while True:
        r2 = r.replace(vx(k_[1], k_[2], k_[3], mu_[1], mu_[2], mu_[3])*x___,
                           (- d(mu_[1], mu_[3]) * d(k_[1], mu_[2])
                            + d(mu_[1], mu_[2]) * d(k_[1], mu_[3])
                            + d(mu_[2], mu_[3]) * d(k_[2], mu_[1])
                            - d(mu_[1], mu_[2]) * d(k_[2], mu_[3])
                            - d(mu_[2], mu_[3]) * d(k_[3], mu_[1])
                            + d(mu_[1], mu_[3]) * d(k_[3], mu_[2])
                            )*x___
                           ).expand()

        if r == r2:
            return r

        r2 = r2.replace(d(mu_[0], mu_[1])*d(mu_[0], mu_[2]),
                            d(mu_[1], mu_[2]), ~mu_[0].req_type(AtomType.Fn), level_range=(0, 0), repeat=True)

        r2 = r2.replace(d(mu_[0], mu_[0]), 4, ~mu_[
            0].req_type(AtomType.Fn), level_range=(0, 0), repeat=True)

        r2 = r2.replace(d(k(x_), k(y_)), k(x_, y_),
                            level_range=(0, 0), repeat=True)
        r2 = r2.replace(d(mu_[0], k(y_))**2, k(y_, y_), ~mu_[0].req_type(AtomType.Fn),
                            level_range=(0, 0), repeat=True)
        r2 = r2.replace(d(mu_[0], mu_[1])**2, 4, ~mu_[0].req_type(AtomType.Fn) & ~mu_[1].req_type(AtomType.Fn),  # 4 instead of D
                            level_range=(0, 0), repeat=True)
        print('Terms', len(r2))

        r = r2

gluon(r)
#print('\t', gluon(r).format(terms_on_new_line=True))

Terms 9
Terms 72
Terms 560
Terms 2654
Terms 3541
Terms 6044
Terms 10516
Terms 9652


-12*mom(0,0)^2*mom(2,3)^2*d(nu1,nu2)+55*mom(0,1)^2*mom(0,3)^2*d(nu1,nu2)-7*mom(0,1)^2*mom(3,3)^2*d(nu1,nu2)-16*mom(0,1)^2*mom(3,4)^2*d(nu1,nu2)+100*mom(0,2)^2*mom(0,3)^2*d(nu1,nu2)+4*mom(0,2)^2*mom(0,4)^2*d(nu1,nu2)-6*mom(0,2)^2*mom(3,3)^2*d(nu1,nu2)-28*mom(0,2)^3*mom(0,3)*d(nu1,nu2)+44*mom(0,2)^3*mom(0,4)*d(nu1,nu2)-12*mom(0,2)^3*mom(3,3)*d(nu1,nu2)-4*mom(0,2)^3*mom(3,4)*d(nu1,nu2)-28*mom(0,2)^3*mom(4,4)*d(nu1,nu2)+138*mom(0,2)^3*d(nu1,mom(0))*d(nu2,mom(4))-225*mom(0,2)^3*d(nu1,mom(1))*d(nu2,mom(4))+102*mom(0,2)^3*d(nu1,mom(2))*d(nu2,mom(4))+24*mom(0,2)^3*d(nu1,mom(3))*d(nu2,mom(0))-12*mom(0,2)^3*d(nu1,mom(3))*d(nu2,mom(3))-44*mom(0,2)^3*d(nu1,mom(3))*d(nu2,mom(4))-36*mom(0,2)^3*d(nu1,mom(4))*d(nu2,mom(0))+116*mom(0,2)^3*d(nu1,mom(4))*d(nu2,mom(3))-36*mom(0,2)^3*d(nu1,mom(4))*d(nu2,mom(4))+6*mom(0,3)^2*mom(1,2)^2*d(nu1,nu2)-6*mom(0,3)^2*mom(2,2)^2*d(nu1,nu2)+8*mom(0,3)^3*mom(1,1)*d(nu1,nu2)-58*mom(0,3)^3*mom(1,2)*d(nu1,nu2)-12*mom(0,3)^3*mom(2,2)*d(nu1,nu2)-24*mom(0,3)^3*d(nu1,mom(0))

In [57]:
def gluon_transformer():
    return Transformer().repeat(
        Transformer().stats('gluon',
                            Transformer().replace(
                                vx(k_[1], k_[2], k_[3], mu_[
                                    1], mu_[2], mu_[3])*x__,
                                (- d(mu_[1], mu_[3]) * d(k_[1], mu_[2])
                                 + d(mu_[1], mu_[2]) *
                                    d(k_[1], mu_[3])
                                    + d(mu_[2], mu_[3]) * d(k_[2], mu_[1])
                                    - d(mu_[1], mu_[2]) * d(k_[2], mu_[3])
                                    - d(mu_[2], mu_[3]) * d(k_[3], mu_[1])
                                    + d(mu_[1], mu_[3]) * d(k_[3], mu_[2]))*x__,
                                level_range=(0, 0)
                            ).expand().
                            repeat(Transformer().replace(
                                d(mu_[0], mu_[1]) *
                                d(mu_[0], mu_[2]),
                                d(mu_[1], mu_[2]), ~mu_[
                                    0].req_type(AtomType.Fn),
                                level_range=(0, 0)
                            ).replace(
                                d(k(x_), k(y_)),
                                k(x_, y_),
                                level_range=(0, 0)
                            ).replace(
                                d(mu_[0], k(y_))**2,
                                k(y_, y_),
                                ~mu_[0].req_type(
                                    AtomType.Fn),
                                level_range=(0, 0)
                            ).replace(
                                d(mu_[0], mu_[1])**2,
                                4,
                                ~mu_[0].req_type(AtomType.Fn) & ~mu_[
                                    1].req_type(AtomType.Fn),
                                level_range=(0, 0)
                            ))).check_interrupt()
    )


e = r.transform().chain(gluon_transformer()).execute()

Stats for gluon:
	In  │ 1 │ 769.00 B │
	Out │ 9 │   6.03KB │ ⧗ 139.75µs
Stats for gluon:
	In  │  9 │   6.03KB │
	Out │ 72 │  41.30KB │ ⧗ 3.47ms
Stats for gluon:
	In  │  72 │  41.30KB │
	Out │ 560 │ 266.76KB │ ⧗ 14.05ms
Stats for gluon:
	In  │  560 │ 266.76KB │
	Out │ 2654 │   1.08MB │ ⧗ 64.69ms
Stats for gluon:
	In  │ 2654 │   1.08MB │
	Out │ 3541 │   1.16MB │ ⧗ 273.99ms
Stats for gluon:
	In  │ 3541 │   1.16MB │
	Out │ 6044 │   1.50MB │ ⧗ 340.87ms
Stats for gluon:
	In  │  6044 │   1.50MB │
	Out │ 10516 │   1.76MB │ ⧗ 502.35ms
Stats for gluon:
	In  │ 10516 │   1.76MB │
	Out │  9652 │ 851.32KB │ ⧗ 573.17ms
Stats for gluon:
	In  │ 9652 │ 851.32KB │
	Out │ 9652 │ 851.32KB │ ⧗ 94.00ms


In [58]:
def gluon_stream(r):
    s = TermStreamer('gluon', n_cores=1)
    s.push(r)

    for _ in range(10):
        s = s.map(Transformer().replace(
            vx(k_[1], k_[2], k_[3], mu_[
                1], mu_[2], mu_[3])*x__,
            (- d(mu_[1], mu_[3]) * d(k_[1], mu_[2])
             + d(mu_[1], mu_[2]) * d(k_[1], mu_[3])
             + d(mu_[2], mu_[3]) * d(k_[2], mu_[1])
             - d(mu_[1], mu_[2]) * d(k_[2], mu_[3])
             - d(mu_[2], mu_[3]) * d(k_[3], mu_[1])
             + d(mu_[1], mu_[3]) * d(k_[3], mu_[2])
             )
            * x__,
            level_range=(0, 0)
        ).expand())

        # s.normalize()

        s = s.map(Transformer().repeat(Transformer().replace(
            d(mu_[0], mu_[1]) *
            d(mu_[0], mu_[2]),
            d(mu_[1], mu_[2]), ~mu_[
                0].req_type(AtomType.Fn),
            level_range=(0, 0)
        ).replace(
            d(k(x_), k(y_)),
            k(x_, y_),
            level_range=(0, 0)
        ).replace(
            d(mu_[0], k(y_))**2,
            k(y_, y_),
            ~mu_[0].req_type(
                AtomType.Fn),
            level_range=(0, 0)
        ).replace(
            d(mu_[0], mu_[1])**2,
            4,
            ~mu_[0].req_type(AtomType.Fn) & ~mu_[
                1].req_type(AtomType.Fn),
            level_range=(0, 0)
        )))

        s.normalize()

    r = s.to_expression()
    print(r.format(terms_on_new_line=True))
    print(len(r))

## Trace4

In 4 dimensions:
$$\gamma(\mu_1,\mu_2,\mu_3,\ldots) = \epsilon^{\mu_1 \mu_2 \mu_3 \nu} \gamma_5 \gamma(\nu,\ldots) + \gamma(\mu_1,\ldots) g(\mu_2,\mu_3) - \gamma(\mu_2,\ldots) g(\mu_1,\mu_3) + \gamma(\mu_3,\ldots) g(\mu_1,\mu_2)$$

- Repeatedly applying this rule (with unique dummy index $\nu$) yields an expression in terms of Levi-Civita symbols and metrics
- The Levi-Civita symbol is fully antisymmetric: in 4D we can only have 4 unique indices. Thus input with many dummy indices will have many canceling combinations

In [59]:
import random
from itertools import permutations
import time

T = Transformer()
mu = S('mu', 'nu')
i = S(*['mu<{}>'.format(i) for i in range(21)])

mu = S(*['mu<{}>'.format(i) for i in range(21)])
nu = S(*['nu<{}>'.format(i) for i in range(21)])  # dummy
muw = S(*['mu{}_'.format(i) for i in range(21)])
g5 = S('g5')
levi, sign = S('levi', 'sign', is_antisymmetric=True)
d = S('ds', is_symmetric=True)
gamma = S('gamma', is_cyclesymmetric=True)
x___, y___, D = S('x___', 'y___', 'D')


def levi_contract(l: list[Expression], r: list[Expression]) -> Expression:
    a = [(tuple(zip(l, perm)), sign(*perm).replace(sign(x___), 1)) for perm in permutations(r)]

    res = N(0)
    for (c, sig) in a:
        s = sig
        for b in c:
            s = s * d(b[0], b[1])
        res = res + s

    return res


K = 16
e = 4*gamma(*mu[:K])

t = time.time()

for i in range(K):
    e = e.replace(
        gamma(muw[1], muw[2], muw[3], x___),
        levi(muw[1], muw[2], muw[3], nu[i])*g5*gamma(nu[i], x___) +
        gamma(muw[1], x___)*d(muw[2], muw[3]) - gamma(muw[2], x___)*d(muw[1], muw[3]) + gamma(muw[3], x___)*d(muw[1], muw[2])).expand()
    e = e.replace(g5**2, 1)
print('yo', time.time() - t)

e = e.replace(g5*gamma(x___), gamma(g5, x___))
e = e.replace(gamma(muw[0], muw[1], muw[2]), 0)
e = e.replace(gamma(muw[0], muw[1]), d(muw[0], muw[1]), repeat=True)

print('yo1', time.time() - t)

# TODO: break and continue statement in loops?, IfMatch, Jump?
t = T.repeat(
    T.repeat(T.replace(
        d(muw[0], muw[1])*d(muw[0], muw[2]), d(muw[1], muw[2]))),
    T.replace(d(muw[0], muw[0]), 4),
    T.repeat(T.replace(
        levi(muw[0], x___)*d(muw[0], muw[1]), sign(muw[0], x___)*levi(muw[1], x___,))),
    T.replace(levi(muw[0], muw[1], muw[6], muw[7])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[6], muw[7])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1]], [muw[4], muw[5]])),
    T.replace(levi(muw[0], muw[1], muw[2], muw[7])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[2], muw[7])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1], muw[2]], [muw[4], muw[5], muw[6]])),
    T.replace(levi(muw[0], muw[1], muw[2], muw[3])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[2], muw[3])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1], muw[2], muw[3]], [muw[4], muw[5], muw[6], muw[7]])),
    T.replace(sign(x___), 1),
    T.expand()
)

e = t(e)
print(len(e))


TypeError: Symbol python::gamma redefined with new attributes.

In [ ]:

t = T.repeat(
    T.repeat(T.replace(
        d(muw[0], muw[1])*d(muw[0], muw[2]), d(muw[1], muw[2])).replace(
        d(muw[0], muw[1])**2,4)),
    T.replace(d(muw[0], muw[0]), 4),
    T.repeat(T.replace(
        levi(muw[0], x___)*d(muw[0], muw[1]), sign(muw[0], x___)*levi(muw[1], x___))),
    T.replace(levi(muw[0], muw[1], muw[2], muw[3])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[2], muw[3])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1], muw[2], muw[3]], [muw[4], muw[5], muw[6], muw[7]])),
    T.replace(sign(x___), 1),
    T.expand(),
)

e2 = t(e2)
print(len(e2))

# print(e.format(terms_on_new_line=True))
# print(e2.format(terms_on_new_line=True))

# evaluate
vals = [[random.uniform(0, 1) for j in range(4)]
        for i in range(K)]


for x in range(K):
    for y in range(x + 1, K):
        e = e.replace(d(mu[x], mu[y]), sum(
            [vals[x][i] * vals[y][i] for i in range(4)]))
        e2 = e2.replace(d(mu[x], mu[y]), sum(
            [vals[x][i] * vals[y][i] for i in range(4)]))

print(e)
print(e2)

## Old

In [ ]:
import copy
import random

T = Transformer()
mu = S(*['mu<{}>'.format(i) for i in range(21)])
nu = S(*['nu<{}>'.format(i) for i in range(21)])  # dummy
muw = S(*['mu{}_'.format(i) for i in range(21)])
g5 = S('g5')
levi, sign = S('levi', 'sign', is_antisymmetric=True)
d = S('d', is_symmetric=True)
gamma = S('gamma', is_cyclesymmetric=True)
x___, y___, D = S('x___', 'y___', 'D')


def levi_contract(l: list[Expression], r: list[Expression]) -> Expression:
    from itertools import permutations
    a = [(tuple(zip(l, perm)), sign(*perm)) for perm in permutations(r)]

    res = N(0)
    for t in a:
        s = t[1]
        for b in t[0]:
            s = s * d(b[0], b[1])
        res = res + s

    return res.replace(sign(x___), 1)


K = 4
e = 4*gamma(*mu[:K])


for i in range(K):
    e = e.replace(
        gamma(muw[1], muw[2], muw[3], x___),
        levi(muw[1], muw[2], muw[3], nu[i])*g5*gamma(nu[i], x___) +
        gamma(muw[1], x___)*d(muw[2], muw[3]) - gamma(muw[2], x___)*d(muw[1], muw[3]) + gamma(muw[3], x___)*d(muw[1], muw[2])).expand()
    e = e.replace(g5**2, 1, repeat=True)

e = e.replace(g5*gamma(x___), gamma(g5, x___))
e = e.replace(gamma(muw[0], muw[1], muw[2]), 0, repeat=True)
e = e.replace(gamma(muw[0], muw[1]), d(muw[0], muw[1]), repeat=True)


e2 = copy.deepcopy(e)

# TODO: break and continue statement in loops?, IfMatch, Jump?
t = T.repeat(
    T.repeat(T.replace(
        d(muw[0], muw[1])*d(muw[0], muw[2]), d(muw[1], muw[2]))),
    T.replace(d(muw[0], muw[0]), 4),
    T.repeat(T.replace(
        levi(muw[0], x___)*d(muw[0], muw[1]), sign(muw[0], x___)*levi(muw[1], x___,))),
    T.replace(levi(muw[0], muw[1], muw[6], muw[7])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[6], muw[7])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1]], [muw[4], muw[5]])),
    T.replace(levi(muw[0], muw[1], muw[2], muw[7])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[2], muw[7])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1], muw[2]], [muw[4], muw[5], muw[6]])),
    T.replace(levi(muw[0], muw[1], muw[2], muw[3])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[2], muw[3])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1], muw[2], muw[3]], [muw[4], muw[5], muw[6], muw[7]])),
    T.replace(sign(x___), 1),
    T.expand()
)

e = t(e)
print(len(e))

t = T.repeat(
    T.repeat(T.replace(
        d(muw[0], muw[1])*d(muw[0], muw[2]), d(muw[1], muw[2]))),
    T.replace(d(muw[0], muw[0]), 4),
    T.repeat(T.replace(
        levi(muw[0], x___)*d(muw[0], muw[1]), sign(muw[0], x___)*levi(muw[1], x___))),
    T.replace(levi(muw[0], muw[1], muw[2], muw[3])*levi(muw[4], muw[5], muw[6], muw[7]),
                  sign(muw[0], muw[1], muw[2], muw[3])*sign(muw[4], muw[5], muw[6], muw[7]) *
                  levi_contract([muw[0], muw[1], muw[2], muw[3]], [muw[4], muw[5], muw[6], muw[7]])),
    T.replace(sign(x___), 1),
    T.expand(),
)

e2 = t(e2)
print(len(e2))

# print(e.format(terms_on_new_line=True))
# print(e2.format(terms_on_new_line=True))

# evaluate
vals = [[random.uniform(0, 1) for j in range(4)]
        for i in range(K)]


for x in range(K):
    for y in range(x + 1, K):
        e = e.replace(d(mu[x], mu[y]), sum(
            [vals[x][i] * vals[y][i] for i in range(4)]))
        e2 = e2.replace(d(mu[x], mu[y]), sum(
            [vals[x][i] * vals[y][i] for i in range(4)]))

print(e)
print(e2)
